# LCC PEMWE

## 1. Inputs and model setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
from inputs import (
    ELECTRICITY_HISTORY,
    ELECTRICITY_HISTORY_NO_WAR,
    WATER_HISTORY,

    PEMWE_MATERIALS,
    PEMWE_CONSTRUCTION_LABOUR_FRACTION,
    # INPUT_SOURCES,

    LIFETIME_YEARS,
    INFLATION_RATE,
    EQUITY,
    EQUITY_RETURN,
    DEBT,
    DEBT_INTEREST_RATE,

    OPEX_LABOUR_FRACTION,
    PEMWE_MAINTENANCE_COST,

    PEMWE_CAPACITY_MW,
    PEMWE_H2_PER_HALF_YEAR,
    PEMWE_ELECTRICITY_KWH_PER_KG,
    PEMWE_WATER_KG_PER_KG,
    PEMWE_KOH_KG_PER_KG,
    PEMWE_KOH_PRICE_EUR_PER_KG,
)
from util_model import (
    material_cost_table,
    investment_cost,
)

from util_plotting import (
    plot_timeseries,
    plot_cost_breakdown,
    plot_distribution,
    plot_sobol, 
)

# Working copies
df_elc = ELECTRICITY_HISTORY.copy()
df_elc_nw = ELECTRICITY_HISTORY_NO_WAR.copy()
df_water = WATER_HISTORY.copy()


# PD Display format for numbers
def sane_number(x):
    if pd.isna(x):
        return ""
    if abs(x) < 1:
        return f"{x: .6f}"       # 0.003656, 0.131125
    elif float(x).is_integer():
        return f"{x: .0f}"       # 489 700 / 25 856 160
    else:
        return f"{x: .2f}"       # 17 905.32 / 29.53

pd.options.display.float_format = sane_number

In [3]:
datasets = {
    "Electricity": df_elc,
    "Electricity — no-war baseline": df_elc_nw,
    "Water": df_water,
}

data_overview = pd.DataFrame({
    name: {
        "Rows": len(df),
        "Columns": len(df.columns),
        "Start": df["DATE"].min(),
        "End": df["DATE"].max(),
        "Missing": int(df.isna().sum().sum()),
        "Duplicates": int(df.duplicated().sum()),
    }
    for name, df in datasets.items()
})

print(data_overview)


# Quick plots of historical inputs
hist_electricity_plot = plot_timeseries(
    df_elc,
    y="no_tax",
    title="Historical electricity price",
    y_title="EUR/kWh",
)

hist_electricity_nw_plot = plot_timeseries(
    df_elc_nw,
    y="no_tax",
    title="Historical electricity price — no-war baseline",
    y_title="EUR/kWh",
)

hist_water_plot = plot_timeseries(
    df_water,
    y=["WF", "WWF"],
    title="Historical water and wastewater prices",
    y_title="EUR/kg",
)


hist_water_plot.show()
hist_electricity_nw_plot.show()
hist_electricity_plot.show()

                    Electricity Electricity — no-war baseline  \
Rows                         44                            29   
Columns                       2                             2   
Start       2004-01-31 00:00:00           2004-01-31 00:00:00   
End         2025-07-31 00:00:00           2018-01-31 00:00:00   
Missing                       0                             0   
Duplicates                    0                             0   

                          Water  
Rows                         36  
Columns                       3  
Start       2008-01-31 00:00:00  
End         2025-07-31 00:00:00  
Missing                       0  
Duplicates                    0  


## 2. Material and component cost

### 2.1 Deterministic material cost

In [4]:
# 4. PEMWE material cost / CAPEX

material_df = material_cost_table(PEMWE_MATERIALS)

material_cost = material_df["Cost"].sum()


# Sanity checks
assert material_df["Cost"].ge(0).all()
assert np.isclose(
    material_df["Cost Share"].sum(),
    1.0,
)


display(
    material_df[["Material", "Unit Price", "Amount", "Cost", "Cost Share"]]
    .sort_values(
        "Cost",
        ascending=False,
    )
    .head(10)
)

plot_cost_breakdown(
    material_df,
    title="PEMWE material cost breakdown",
).show()

,Material,Unit Price,Amount,Cost,Cost Share
4,Iridium,57171.74,9.86,563713.40,0.428261
15,Rectifier / Power Electronics,320699,1,320699,0.243639
6,Nafion N117,1800,88.54,159372,0.121077
14,Pumps,9300,6,55800,0.042392
8,Stack Cooling Heat Exchanger,48000,1,48000,0.036466
2,Titanium,11.93,3987,47544.98,0.036121
9,Condenser,42000,1,42000,0.031908
5,Carbon Paper / GDL,80.62,329,26523.98,0.020151
3,Platinum,31263.02,0.670000,20946.22,0.015913
17,Housing,4950,2,9900,0.007521


### 2.2 Material-cost uncertainty

> ### Distribution assumptions
> Historical metal prices use empirical uncertainty distributions: Steel and Copper are triangular, while Nickel is lognormal.

In [ ]:
from util_uncertainty import material_uncertainty_table, run_material_monte_carlo
from inputs import HISTORICAL_METAL_STATS

material_uncertainty = material_uncertainty_table(
    material_df,
    HISTORICAL_METAL_STATS,
)

material_uncertainty[
    [
        "Material",
        "Unit Price",
        "Cost",
        "Cost Share",
        "Distribution",
        "Relative Uncertainty",
    ]
]

# run mc

material_mc = run_material_monte_carlo(
    material_uncertainty,
    HISTORICAL_METAL_STATS,
    n_draws=2048 * 5, 
    seed=123456,
)
material_cost_samples = np.asarray(
    material_mc.outvars["Material Cost"].vals,
    dtype=float,
)


display(pd.Series({
    "Mean": material_cost_samples.mean(),
    "Std": material_cost_samples.std(ddof=1),
    "P05": np.quantile(material_cost_samples, 0.05),
    "Median": np.quantile(material_cost_samples, 0.50),
    "P95": np.quantile(material_cost_samples, 0.95),
}))


material_cost_samples = np.asarray(
    material_mc.outvars["Material Cost"].vals,
    dtype=float,
)

plot_distribution(
    material_cost_samples,
    title="PEMWE material cost uncertainty",
    x_title="Material cost [EUR]",
).show()

Mean      1515352.62
Std        318637.87
P05       1059507.15
Median    1479042.14
P95       2096082.48
dtype: float64

### 2.3 Material-cost sensitivity

>
> OLD Sobol:
> hard-coded structural baselines different from deterministic EoL
>
> NEW Sobol:
> actual deterministic EoL baselines ±10% around those exact values
> The old Sobol analysis instead used roughly:
> 
> transport = 7250
> salvage = 14500
> construction = MODEL_DISTRIBUTIONS["AWE_CAPEX"]["mean"]
> 
> So the large change comes especially from:
> 
> salvage:
> old Sobol baseline    14,500 EUR
> new Sobol baseline    4,257 EUR
> 
> and also:
> 
> construction:
> old Sobol baseline    1.343 M EUR
> new baseline          0.929 M EUR

In [6]:
from SALib.sample import sobol as sobol_sample
from SALib.analyze import sobol as sobol_analyze

def build_equal_range_material_problem(
    material_df,
    relative_half_width,
):
    names = material_df["Material"].tolist()
    baseline_prices = material_df["Unit Price"].to_numpy(dtype=float)

    bounds = [
        [
            price * (1.0 - relative_half_width),
            price * (1.0 + relative_half_width),
        ]
        for price in baseline_prices
    ]

    return {
        "num_vars": len(names),
        "names": names,
        "bounds": bounds,
        "dists": ["unif"] * len(names),
    }

# Structural Sobol sensitivity analysis of material cost.
#
# Three common uncertainty ranges are evaluated:
#   +-10% -> main analysis
#   +-20% -> robustness check
#   +-25% -> robustness check
#
# The material-cost model is linear in the unit prices, so
# second-order interaction indices are not required.

material_sobol = {}

amounts = material_df["Amount"].to_numpy(dtype=float)

for half_width in [0.10, 0.20, 0.25]:

    problem = build_equal_range_material_problem(
        material_df,
        relative_half_width=half_width,
    )

    samples = sobol_sample.sample(
        problem,
        N=2048,
        calc_second_order=False,
        scramble=True,
        seed=123456
    )

    Y = samples @ amounts

    Si = sobol_analyze.analyze(
        problem,
        Y,
        calc_second_order=False,
        print_to_console=False,
    )

    material_sobol[half_width] = {
        "problem": problem,
        "Si": Si,
    }

In [7]:
from util_plotting import plot_sobol_results

material_sobol_results, material_sobol_figures = (
    plot_sobol_results(
        material_sobol[0.20]["problem"],
        material_sobol[0.20]["Si"],
        technology="PEMWE",
        dominant_threshold=0.05,
        minor_axis_floor=0.015,
        save_path="plots/PEMWE_material_sobol"
    )
)

for fig in material_sobol_figures.values():
    fig.show()

## 3. Capital expenditure

### 3.1 Deterministic CAPEX

In [8]:

# Calculate deterministic CAPEX from the baseline material cost.
capex = investment_cost(
    material_cost=material_cost,
    labour_fraction=PEMWE_CONSTRUCTION_LABOUR_FRACTION,
)

# Display results
for c in capex:
    print(f"{c}: EUR {capex[c]:,.2f}")



material_cost: EUR 1,316,285.88
labour_cost: EUR 273,787.46
investment_grant: EUR 0.00
capex: EUR 1,590,073.34


### 3.2 Derived CAPEX uncertainty

In [9]:
# Propagate the material-cost Monte Carlo samples into CAPEX.
#
# Construction labour is defined as a fixed fraction of material cost.
# Therefore, each sampled material cost produces a corresponding
# construction-labour cost and total CAPEX value.

construction_labour_samples = (
    material_cost_samples
    * PEMWE_CONSTRUCTION_LABOUR_FRACTION
)

capex_samples = (
    material_cost_samples
    + construction_labour_samples
)

# Plot the resulting CAPEX distribution.
fig = plot_distribution(
    capex_samples,
    title="PEMWE CAPEX uncertainty",
    x_title="CAPEX [EUR]",
)

fig.show()

# 4. Resource-price scenarios

## 4.1 High electricity-price scenario (S1)

In [10]:
# S1 Electricity Forecast
from util_forecasting import prophet_forecast
from util_plotting import plot_forecast

electricity_s1 = prophet_forecast(
    df_elc,
    value_col="no_tax",
    years=20,
)

electricity_s1 = electricity_s1.rename(columns={
    "Lower": "S1 Lower",
    "Mean": "S1 Mean",
    "Upper": "S1 Upper",
})

display(electricity_s1.head(10))

plot_forecast(
    df_elc,
    electricity_s1,
    history_col="no_tax",
    mean_col="S1 Mean",
    lower_col="S1 Lower",
    upper_col="S1 Upper",
    title="PEMWE electricity price forecast — Scenario 1",
    y_title="EUR/kWh",
).show()

s:\AWSI\GreenTech\JOULE - LCC Paper\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
19:48:09 - cmdstanpy - INFO - Chain [1] start processing
19:48:10 - cmdstanpy - INFO - Chain [1] done processing


,DATE,S1 Lower,S1 Mean,S1 Upper
0,2025-07-31,0.126626,0.152161,0.178247
1,2026-01-31,0.130704,0.156005,0.180120
2,2026-07-31,0.134524,0.158211,0.182864
3,2027-01-31,0.137141,0.160060,0.182754
4,2027-07-31,0.136232,0.164183,0.188119
5,2028-01-31,0.137169,0.163215,0.188107
6,2028-07-31,0.136825,0.161070,0.186034
7,2029-01-31,0.141778,0.166084,0.191733
8,2029-07-31,0.142501,0.167196,0.192504
9,2030-01-31,0.146086,0.171039,0.194907


## 4.2 Reference electricity-price scenario (S2)

In [11]:
# S2 assumes a discrete regime change after the initial
# three-year period: from 2029 onward, prices follow the
# forecast fitted to the pre-war historical series.
electricity_s2_initial = prophet_forecast(
    df_elc,
    value_col="no_tax",
    years=3,
)



electricity_s2_normal = prophet_forecast(
    df_elc_nw,
    value_col="no_tax",
    years=20,
)


# Combine the two forecasts into a single dataframe
s2_transition_date = electricity_s2_initial["DATE"].max()

electricity_s2_rest = electricity_s2_normal[
    electricity_s2_normal["DATE"] > s2_transition_date
].copy()

electricity_s2 = pd.concat(
    [
        electricity_s2_initial,
        electricity_s2_rest,
    ],
    ignore_index=True,
)

electricity_s2 = electricity_s2.rename(columns={
    "Lower": "S2 Lower",
    "Mean": "S2 Mean",
    "Upper": "S2 Upper",
})

display(electricity_s2.head(10))
plot_forecast(
    df_elc,
    electricity_s2,
    history_col="no_tax",
    mean_col="S2 Mean",
    lower_col="S2 Lower",
    upper_col="S2 Upper",
    title="PEMWE electricity price forecast — Scenario 2",
    y_title="EUR/kWh",
).show()

19:48:10 - cmdstanpy - INFO - Chain [1] start processing
19:48:11 - cmdstanpy - INFO - Chain [1] done processing
19:48:11 - cmdstanpy - INFO - Chain [1] start processing
19:48:11 - cmdstanpy - INFO - Chain [1] done processing


,DATE,S2 Lower,S2 Mean,S2 Upper
0,2025-07-31,0.127647,0.152161,0.174849
1,2026-01-31,0.131125,0.156005,0.180277
2,2026-07-31,0.134664,0.158211,0.183774
3,2027-01-31,0.134830,0.160060,0.185169
4,2027-07-31,0.140170,0.164183,0.189195
5,2028-01-31,0.140048,0.163215,0.186007
6,2028-07-31,0.135538,0.161070,0.187388
7,2029-01-31,0.110748,0.121777,0.132034
8,2029-07-31,0.109953,0.121678,0.133373
9,2030-01-31,0.112223,0.123820,0.134773


In [12]:
assert len(electricity_s1) == 41
assert len(electricity_s2) == 41

assert electricity_s1["DATE"].equals(
    electricity_s2["DATE"]
)

electricity_s2[
    electricity_s2["DATE"].between(
        "2027-07-31",
        "2029-07-31",
    )
]

,DATE,S2 Lower,S2 Mean,S2 Upper
4,2027-07-31,0.140170,0.164183,0.189195
5,2028-01-31,0.140048,0.163215,0.186007
6,2028-07-31,0.135538,0.161070,0.187388
7,2029-01-31,0.110748,0.121777,0.132034
8,2029-07-31,0.109953,0.121678,0.133373


In [13]:
import plotly.graph_objects as go

fig = go.Figure()

# Historical electricity price.
fig.add_trace(go.Scatter(
    x=df_elc["DATE"],
    y=df_elc["no_tax"],
    mode="lines",
    name="Historical",
    line={"color": "gray"},
))

# =====================================================
# S1 uncertainty range
# =====================================================

fig.add_trace(go.Scatter(
    x=electricity_s1["DATE"],
    y=electricity_s1["S1 Lower"],
    mode="lines",
    line={"width": 0},
    showlegend=False,
    hoverinfo="skip",
))

fig.add_trace(go.Scatter(
    x=electricity_s1["DATE"],
    y=electricity_s1["S1 Upper"],
    mode="lines",
    line={"width": 0},
    fill="tonexty",
    fillcolor="rgba(47, 107, 154, 0.18)",
    name="S1 range",
    hoverinfo="skip",
))

fig.add_trace(go.Scatter(
    x=electricity_s1["DATE"],
    y=electricity_s1["S1 Mean"],
    mode="lines",
    name="S1 — High-price scenario",
    line={"color": "#2F6B9A"},
))

# =====================================================
# S2 uncertainty range
# =====================================================

fig.add_trace(go.Scatter(
    x=electricity_s2["DATE"],
    y=electricity_s2["S2 Lower"],
    mode="lines",
    line={"width": 0},
    showlegend=False,
    hoverinfo="skip",
))

fig.add_trace(go.Scatter(
    x=electricity_s2["DATE"],
    y=electricity_s2["S2 Upper"],
    mode="lines",
    line={"width": 0},
    fill="tonexty",
    fillcolor="rgba(217, 119, 6, 0.18)",
    name="S2 range",
    hoverinfo="skip",
))

fig.add_trace(go.Scatter(
    x=electricity_s2["DATE"],
    y=electricity_s2["S2 Mean"],
    mode="lines",
    name="S2 - Reference scenario",
    line={"color": "#D97706"},
))

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Electricity price [EUR/kWh]",
    font={
        "size": 16,
    },
    margin={
        "t": 25,
        "r": 20,
        "b": 20,
        "l": 20,
    },
    legend={
        "orientation": "h",
        "x": 0.1,
        "xanchor": "left",
        "y": -0.15,
        "yanchor": "top",
        "font": {
            "size": 14,
        },
    },
)

fig.write_image(
    "plots/electricity_price_scenarios.svg",
    format="svg",
)


fig.show()

## 4.3 Water-price trajectory

In [14]:
# Water and wastewater fee forecast

water_fee_forecast = prophet_forecast(
    df_water,
    value_col="WF",
    years=20,
)

wastewater_fee_forecast = prophet_forecast(
    df_water,
    value_col="WWF",
    years=20,
)

water_forecast = pd.DataFrame({
    "DATE": water_fee_forecast["DATE"],

    "Water Lower": (
        water_fee_forecast["Lower"]
        + wastewater_fee_forecast["Lower"]
    ),

    "Water Mean": (
        water_fee_forecast["Mean"]
        + wastewater_fee_forecast["Mean"]
    ),

    "Water Upper": (
        water_fee_forecast["Upper"]
        + wastewater_fee_forecast["Upper"]
    ),
})

display(water_forecast.head(10))
df_water["Total"] = (
    df_water["WF"]
    + df_water["WWF"]
)



plot_forecast(
    df_water,
    water_forecast,
    history_col="Total",
    mean_col="Water Mean",
    lower_col="Water Lower",
    upper_col="Water Upper",
    title="Water price forecast",
    y_title="EUR/kg",
).show()

19:48:13 - cmdstanpy - INFO - Chain [1] start processing
19:48:13 - cmdstanpy - INFO - Chain [1] done processing
19:48:13 - cmdstanpy - INFO - Chain [1] start processing
19:48:13 - cmdstanpy - INFO - Chain [1] done processing


,DATE,Water Lower,Water Mean,Water Upper
0,2025-07-31,0.003728,0.003833,0.003941
1,2026-01-31,0.003656,0.003761,0.003860
2,2026-07-31,0.003696,0.003794,0.003896
3,2027-01-31,0.003634,0.003730,0.003825
4,2027-07-31,0.003623,0.003736,0.003836
5,2028-01-31,0.003634,0.003746,0.003851
6,2028-07-31,0.003675,0.003777,0.003883
7,2029-01-31,0.003649,0.003761,0.003874
8,2029-07-31,0.003642,0.003757,0.003873
9,2030-01-31,0.003564,0.003684,0.003800


# 5. Resource Costs

In [15]:
# The first forecast point (2025-07-31) is the price anchor.
# The 20-year operating horizon consists of the following
# 40 half-year periods.
from util_model import resource_cost_table

electricity_s1_operating = electricity_s1.iloc[1:].reset_index(drop=True)

electricity_s2_operating = electricity_s2.iloc[1:].reset_index(drop=True)

water_operating = water_forecast.iloc[1:].reset_index(drop=True)


# Sanity checks 
assert len(electricity_s1_operating) == 40
assert len(electricity_s2_operating) == 40
assert len(water_operating) == 40

assert electricity_s1_operating["DATE"].equals(water_operating["DATE"])

assert electricity_s2_operating["DATE"].equals(water_operating["DATE"])


# Calculate the resource costs for both scenarios
resource_s1 = resource_cost_table(
    electricity_s1_operating,
    water_operating,
    electricity_cols=(
        "S1 Lower",
        "S1 Mean",
        "S1 Upper",
    ),
    h2_per_period=PEMWE_H2_PER_HALF_YEAR,
    electricity_kwh_per_kg=PEMWE_ELECTRICITY_KWH_PER_KG,
    water_kg_per_kg=PEMWE_WATER_KG_PER_KG,
    koh_kg_per_kg=PEMWE_KOH_KG_PER_KG,
    koh_price_eur_per_kg=PEMWE_KOH_PRICE_EUR_PER_KG,
)

display(resource_s1.head())


resource_s2 = resource_cost_table(
    electricity_s2_operating,
    water_operating,
    electricity_cols=(
        "S2 Lower",
        "S2 Mean",
        "S2 Upper",
    ),
    h2_per_period=PEMWE_H2_PER_HALF_YEAR,
    electricity_kwh_per_kg=PEMWE_ELECTRICITY_KWH_PER_KG,
    water_kg_per_kg=PEMWE_WATER_KG_PER_KG,
    koh_kg_per_kg=PEMWE_KOH_KG_PER_KG,
    koh_price_eur_per_kg=PEMWE_KOH_PRICE_EUR_PER_KG,
)

display(resource_s2.head())

,DATE,Electricity Price Lower,Electricity Price Mean,Electricity Price Upper,Water Price Lower,Water Price Mean,Water Price Upper,H2 Production,Electricity Consumption,Water Consumption,...,Electricity Cost Lower,Water Cost Lower,Electricity Cost Mean,Water Cost Mean,Electricity Cost Upper,Water Cost Upper,KOH Cost,Total Resource Cost Lower,Total Resource Cost Mean,Total Resource Cost Upper
0,2026-01-31,0.130704,0.156005,0.180120,0.003656,0.003761,0.003860,222455.76,11122787.76,2068838.52,...,1453797.30,7564.47,1735207.58,7780.25,2003440.80,7985.97,0.000000,1461361.77,1742987.83,2011426.78
1,2026-07-31,0.134524,0.158211,0.182864,0.003696,0.003794,0.003896,222455.76,11122787.76,2068838.52,...,1496280.63,7647.12,1759746.03,7849.47,2033960.68,8060.25,0.000000,1503927.75,1767595.50,2042020.93
2,2027-01-31,0.137141,0.160060,0.182754,0.003634,0.003730,0.003825,222455.76,11122787.76,2068838.52,...,1525385.85,7518.73,1780316.57,7716.94,2032736.37,7914.28,0.000000,1532904.58,1788033.51,2040650.65
3,2027-07-31,0.136232,0.164183,0.188119,0.003623,0.003736,0.003836,222455.76,11122787.76,2068838.52,...,1515276.58,7494.86,1826169.39,7730.05,2092403.14,7936.06,0.000000,1522771.44,1833899.43,2100339.20
4,2028-01-31,0.137169,0.163215,0.188107,0.003634,0.003746,0.003851,222455.76,11122787.76,2068838.52,...,1525702.09,7518.84,1815409.18,7749.23,2092273.19,7967.68,0.000000,1533220.93,1823158.40,2100240.87


,DATE,Electricity Price Lower,Electricity Price Mean,Electricity Price Upper,Water Price Lower,Water Price Mean,Water Price Upper,H2 Production,Electricity Consumption,Water Consumption,...,Electricity Cost Lower,Water Cost Lower,Electricity Cost Mean,Water Cost Mean,Electricity Cost Upper,Water Cost Upper,KOH Cost,Total Resource Cost Lower,Total Resource Cost Mean,Total Resource Cost Upper
0,2026-01-31,0.131125,0.156005,0.180277,0.003656,0.003761,0.003860,222455.76,11122787.76,2068838.52,...,1458477.62,7564.47,1735207.58,7780.25,2005179.34,7985.97,0.000000,1466042.09,1742987.83,2013165.32
1,2026-07-31,0.134664,0.158211,0.183774,0.003696,0.003794,0.003896,222455.76,11122787.76,2068838.52,...,1497841.12,7647.12,1759746.03,7849.47,2044079.84,8060.25,0.000000,1505488.24,1767595.50,2052140.09
2,2027-01-31,0.134830,0.160060,0.185169,0.003634,0.003730,0.003825,222455.76,11122787.76,2068838.52,...,1499688.93,7518.73,1780316.57,7716.94,2059592.32,7914.28,0.000000,1507207.66,1788033.51,2067506.60
3,2027-07-31,0.140170,0.164183,0.189195,0.003623,0.003736,0.003836,222455.76,11122787.76,2068838.52,...,1559080.53,7494.86,1826169.39,7730.05,2104379.60,7936.06,0.000000,1566575.39,1833899.43,2112315.66
4,2028-01-31,0.140048,0.163215,0.186007,0.003634,0.003746,0.003851,222455.76,11122787.76,2068838.52,...,1557722.86,7518.84,1815409.18,7749.23,2068912.20,7967.68,0.000000,1565241.70,1823158.40,2076879.88


## 5.1 Scenario resource-cost totals

In [16]:
resource_totals = pd.DataFrame({
    "S1": {
        "Lower": resource_s1[
            "Total Resource Cost Lower"
        ].sum(),
        "Mean": resource_s1[
            "Total Resource Cost Mean"
        ].sum(),
        "Upper": resource_s1[
            "Total Resource Cost Upper"
        ].sum(),
    },

    "S2": {
        "Lower": resource_s2[
            "Total Resource Cost Lower"
        ].sum(),
        "Mean": resource_s2[
            "Total Resource Cost Mean"
        ].sum(),
        "Upper": resource_s2[
            "Total Resource Cost Upper"
        ].sum(),
    },
}).T

display(resource_totals)

,Lower,Mean,Upper
S1,74869738.20,85753257.21,96755730.22
S2,55733504.01,61554675.45,67442978.93


In [17]:
assert len(resource_s1) == 40
assert len(resource_s2) == 40

assert (
    resource_s1[
        "Total Resource Cost Lower"
    ]
    <= resource_s1[
        "Total Resource Cost Mean"
    ]
).all()

assert (
    resource_s1[
        "Total Resource Cost Mean"
    ]
    <= resource_s1[
        "Total Resource Cost Upper"
    ]
).all()

assert (
    resource_s2[
        "Total Resource Cost Lower"
    ]
    <= resource_s2[
        "Total Resource Cost Mean"
    ]
).all()

assert (
    resource_s2[
        "Total Resource Cost Mean"
    ]
    <= resource_s2[
        "Total Resource Cost Upper"
    ]
).all()

In [18]:
lifetime_h2 = (
    resource_s1["H2 Production"].sum()
)

print(
    f"20-year H2 production: "
    f"{lifetime_h2:,.0f} kg"
)

20-year H2 production: 8,898,230 kg


> ##### Hydrogen-production timing
>
> The model uses 444,911.5105 kg H₂ per year.
>
> Resource costs are calculated at half-year resolution, therefore:
>
> - production per period = 222,455.75525 kg H₂,
> - number of periods = 40,
> - lifetime production = 8,898,230.21 kg H₂.

In [19]:
resource_comparison = pd.DataFrame({
    "DATE": resource_s1["DATE"],
    "S1": resource_s1["Total Resource Cost Mean"],
    "S2": resource_s2["Total Resource Cost Mean"],
})


plot_timeseries(
    resource_comparison,
    y=["S1", "S2"],
    title="PEMWE resource cost by operating period",
    y_title="EUR / half-year",
).show()

## 6. End-of-life cost

### 6.1 Deterministic end-of-life cost


In [20]:
from inputs import (
    PEMWE_EOL_ASSUMPTIONS,
    PEMWE_DECONSTRUCTION_FRACTION,
)

from util_model import (
    end_of_life_cost,
    eol_table,
)

# Build the material-level EoL table
# This combines the material inventory with the assumed recycling
# ratios and transport/disposal unit cost
eol_df = eol_table(
    material_df,
    PEMWE_EOL_ASSUMPTIONS,
)


# Aggregate material-level transport/disposal cost and salvage value
transport_disposal_cost = (
    eol_df["Transport Disposal Cost"].sum()
)

salvage_value = (
    eol_df["Salvage Value"].sum()
)


material_cost = material_df["Cost"].sum()

deconstruction =  material_cost * PEMWE_DECONSTRUCTION_FRACTION   

# Evaluate the deterministic EoL model.
#
# The current PEMWE model uses material cost as the construction-cost
# basis for the deconstruction-cost fraction.
eol = end_of_life_cost(
    construction_cost=material_cost,
    deconstruction_fraction=PEMWE_DECONSTRUCTION_FRACTION,
    transport_disposal_cost=transport_disposal_cost,
    salvage_value=salvage_value,
    inflation_rate=INFLATION_RATE,
    equity=EQUITY,
    equity_return=EQUITY_RETURN,
    debt=DEBT,
    debt_interest_rate=DEBT_INTEREST_RATE,
    lifetime_years=LIFETIME_YEARS,
)

real_wacc=eol['real_wacc']
nominal_wacc=eol['nominal_wacc']

# Basic consistency checks.
assert transport_disposal_cost >= 0
assert salvage_value >= 0
assert eol["deconstruction"] >= 0

assert np.isclose(
    eol["nominal_cost"],
    eol["deconstruction"]
    + eol["transport_disposal"]
    - eol["salvage_value"],
)

assert eol["present_cost"] < eol["nominal_cost"]

# Display
print(f"Nominal WACC: {eol['nominal_wacc']:.3%}")
print(f"Real WACC:    {eol['real_wacc']:.3%}")

print("")
print("End-of-life cost summary:")

display(pd.Series(eol))

Nominal WACC: 5.125%
Real WACC:    4.084%

End-of-life cost summary:


deconstruction        78977.15
transport_disposal     7145.04
salvage_value         13997.85
nominal_wacc          0.051250
real_wacc             0.040842
nominal_cost          72124.34
present_cost          32388.38
dtype: float64

### 6.2 End-of-life sensitivity analysis

A structural Sobol sensitivity analysis is applied to the AWE end-of-life submodel. All EoL inputs are varied independently by +-10% around their deterministic baseline values. 

In [21]:
# EoL structural Sobol sensitivity analysis
#
# All EoL inputs are varied independently by +-10% around
# their deterministic baseline values.
#
# Unlike the material-cost model, the EoL model is nonlinear.
# Therefore, second-order Sobol indices are calculated as well.

EOL_SOBOL_HALF_WIDTH = 0.10

eol_baseline = {
    "transport_disposal_cost": transport_disposal_cost,
    "salvage_value": salvage_value,
    "construction_cost": material_cost,
    "deconstruction_fraction": PEMWE_DECONSTRUCTION_FRACTION,
    "inflation_rate": INFLATION_RATE,
    "equity": EQUITY,
    "equity_return": EQUITY_RETURN,
    "debt": DEBT,
    "debt_interest_rate": DEBT_INTEREST_RATE,
}


# Sobol problem definition
eol_names = list(eol_baseline.keys())

eol_bounds = [
    [
        value * (1.0 - EOL_SOBOL_HALF_WIDTH),
        value * (1.0 + EOL_SOBOL_HALF_WIDTH),
    ]
    for value in eol_baseline.values()
]

problem_eol = {
    "num_vars": len(eol_names),
    "names": eol_names,
    "bounds": eol_bounds,
    "dists": ["unif"] * len(eol_names),
}

In [22]:
# Sobol sampling and model evaluation
eol_samples = sobol_sample.sample(
    problem_eol,
    N=2048,
    calc_second_order=True,
    scramble=True,
    seed=123456
)


(
    transport_samples,
    salvage_samples,
    construction_samples,
    deconstruction_fraction_samples,
    inflation_samples,
    equity_samples,
    equity_return_samples,
    debt_samples,
    debt_interest_samples,
) = eol_samples.T

# call the same function
eol_sample_results = end_of_life_cost(
    construction_cost=construction_samples,
    deconstruction_fraction=deconstruction_fraction_samples,
    transport_disposal_cost=transport_samples,
    salvage_value=salvage_samples,
    inflation_rate=inflation_samples,
    equity=equity_samples,
    equity_return=equity_return_samples,
    debt=debt_samples,
    debt_interest_rate=debt_interest_samples,
    lifetime_years=LIFETIME_YEARS,
)

Y_eol = np.asarray(
    eol_sample_results["present_cost"],
    dtype=float,
)

In [23]:
# First-order and total-order sensitivity

Si_eol = sobol_analyze.analyze(
    problem_eol,
    Y_eol,
    calc_second_order=True,
    print_to_console=False,
)

# results table

eol_sobol_results = pd.DataFrame({
    "Parameter": problem_eol["names"],
    "S1": Si_eol["S1"],
    "S1_conf": Si_eol["S1_conf"],
    "ST": Si_eol["ST"],
    "ST_conf": Si_eol["ST_conf"],
})

eol_sobol_results["Interaction_effect"] = (
    eol_sobol_results["ST"]
    - eol_sobol_results["S1"]
)

eol_sobol_results["ST_percent"] = (
    eol_sobol_results["ST"] * 100
)

eol_sobol_results = (
    eol_sobol_results
    .sort_values("ST", ascending=False)
    .reset_index(drop=True)
)

display(eol_sobol_results)

,Parameter,S1,S1_conf,ST,ST_conf,Interaction_effect,ST_percent
0,deconstruction_fraction,0.395020,0.034549,0.397216,0.022922,0.002195,39.72
1,construction_cost,0.394439,0.033824,0.397100,0.022830,0.002661,39.71
2,debt_interest_rate,0.136236,0.020743,0.137117,0.007396,0.000881,13.71
3,equity_return,0.036294,0.010621,0.036967,0.002633,0.000672,3.70
4,inflation_rate,0.013475,0.007719,0.013053,0.000950,-0.000421,1.31
5,salvage_value,0.012396,0.006848,0.012441,0.000802,0.000045,1.24
6,transport_disposal_cost,0.003192,0.003990,0.003242,0.000173,0.000050,0.324152
7,equity,0.002647,0.002962,0.002794,0.000216,0.000146,0.279352
8,debt,0.002370,0.003070,0.002770,0.000200,0.000400,0.276978


In [24]:
S2_eol = pd.DataFrame(
    Si_eol["S2"],
    index=problem_eol["names"],
    columns=problem_eol["names"],
)

display(S2_eol)

# SALib leaves the unused half of the matrix as NaN

interaction_rows = []

names = problem_eol["names"]

for i in range(len(names)):
    for j in range(i + 1, len(names)):

        interaction_rows.append({
            "Parameter 1": names[i],
            "Parameter 2": names[j],
            "S2": Si_eol["S2"][i, j],
            "S2_conf": Si_eol["S2_conf"][i, j],
        })

eol_second_order = (
    pd.DataFrame(interaction_rows)
    .sort_values("S2", ascending=False)
    .reset_index(drop=True)
)

display(eol_second_order)

# Sensitivity plots
eol_sobol_results, eol_sobol_figures = (
    plot_sobol_results(
        problem_eol,
        Si_eol,
        # title="PEMWE end-of-life cost sensitivity",
    )
)

for fig in eol_sobol_figures.values():
    fig.show()

# pd.DataFrame({
#     "New baseline": [
#         transport_disposal_cost,
#         salvage_value,
#         material_cost,  # or whichever construction baseline we use
#         PEMWE_DECONSTRUCTION_FRACTION,
#         INFLATION_RATE,
#         EQUITY,
#         EQUITY_RETURN,
#         DEBT,
#         DEBT_INTEREST_RATE,
#     ],
#     "Old baseline": [
#         7250,
#         14500,
#         "N/A",
#         0.06,
#         0.01,
#         0.15,
#         0.05,
#         0.75,
#         0.045,
#     ],
# }, index=[
#     "transport",
#     "salvage",
#     "construction",
#     "deconstruction fraction",
#     "inflation",
#     "equity",
#     "equity return",
#     "debt",
#     "debt interest",
# ])

,transport_disposal_cost,salvage_value,construction_cost,deconstruction_fraction,inflation_rate,equity,equity_return,debt,debt_interest_rate
transport_disposal_cost,NaN,0.000045,0.000077,0.000044,0.000045,0.000045,0.000052,0.000039,0.000054
salvage_value,NaN,NaN,-0.000067,0.000056,-0.000077,-0.000040,-0.000088,-0.000035,0.000185
construction_cost,NaN,NaN,NaN,0.002067,0.001396,0.000996,0.001296,0.001289,0.001839
deconstruction_fraction,NaN,NaN,NaN,NaN,0.000350,0.000258,0.000507,0.000318,0.000795
inflation_rate,NaN,NaN,NaN,NaN,NaN,-0.001058,-0.001094,-0.001023,-0.001452
equity,NaN,NaN,NaN,NaN,NaN,NaN,0.000053,-0.000045,-0.000017
equity_return,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000408,0.000319
debt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000283
debt_interest_rate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Parameter 1,Parameter 2,S2,S2_conf
0,construction_cost,deconstruction_fraction,0.002067,0.064532
1,construction_cost,debt_interest_rate,0.001839,0.055862
2,construction_cost,inflation_rate,0.001396,0.051812
3,construction_cost,equity_return,0.001296,0.053451
4,construction_cost,debt,0.001289,0.052437
5,construction_cost,equity,0.000996,0.052510
6,deconstruction_fraction,debt_interest_rate,0.000795,0.045636
7,deconstruction_fraction,equity_return,0.000507,0.045007
8,equity_return,debt,0.000408,0.015769
9,deconstruction_fraction,inflation_rate,0.000350,0.045107


In [25]:
from util_plotting import (
    plot_sobol,
    plot_sobol_interaction_matrix,
    plot_sobol_interactions,
)


plot_sobol(
    eol_sobol_results,
    category="Parameter",
    title=(
        "First- and Total-order Sobol Sensitivity "
        "of PEMWE End-of-life Cost"
    ),
).show()


plot_sobol_interactions(
    problem_eol,
    Si_eol,
    top_n=12,
    title=(
        "Largest Second-order Sobol Interactions "
        "in the PEMWE End-of-life Cost Model"
    ),
).show()


plot_sobol_interaction_matrix(
    problem_eol,
    Si_eol,
    title=(
        "Complete Second-order Sobol Interaction Matrix "
        "for PEMWE End-of-life Cost"
    ),
).show()

> ### Baseline consistency note.
> The structural EoL sensitivity analysis uses the same baseline inputs as the deterministic EoL model. This replaces the legacy implementation, which used separate hard-coded baseline values for several EoL sensitivity inputs.

# 7. Operating expenditure

### OPEX model note

The operating-cost calculation currently includes:

- discounted resource cost,
- operating labour, and
- discounted end-of-life cost.

The legacy model also contains a maintenance parameter of EUR 3,750.
Its intended interpretation have not yet been verified, so complete
calculation is used 

In [26]:
from util_model import operation_cost

opex_s1 = {
    bound: operation_cost(
        resource_s1[f"Total Resource Cost {bound}"],
        material_cost=material_cost,
        labour_fraction=OPEX_LABOUR_FRACTION,
        eol_present_cost=eol["present_cost"],
        discount_rate=real_wacc,
    )
    for bound in ["Lower", "Mean", "Upper"]
}

opex_s2 = {
    bound: operation_cost(
        resource_s2[f"Total Resource Cost {bound}"],
        material_cost=material_cost,
        labour_fraction=OPEX_LABOUR_FRACTION,
        eol_present_cost=eol["present_cost"],
        discount_rate=real_wacc,
    )
    for bound in ["Lower", "Mean", "Upper"]
}

opex_results = pd.DataFrame({
    "S1 Lower": opex_s1["Lower"],
    "S1 Mean": opex_s1["Mean"],
    "S1 Upper": opex_s1["Upper"],
    "S2 Lower": opex_s2["Lower"],
    "S2 Mean": opex_s2["Mean"],
    "S2 Upper": opex_s2["Upper"],
}).T

display(opex_results)

,resource_cost,labour_cost,eol_cost,opex
S1 Lower,51015009.39,32907.15,32388.38,51080304.91
S1 Mean,58670468.04,32907.15,32388.38,58735763.57
S1 Upper,66388176.35,32907.15,32388.38,66453471.87
S2 Lower,39040023.23,32907.15,32388.38,39105318.75
S2 Mean,43352910.58,32907.15,32388.38,43418206.11
S2 Upper,47721289.57,32907.15,32388.38,47786585.09


# 8. Total lifecycle cost
TCO combines initial CAPEX with lifecycle OPEX.

In [27]:
tax_impact = 0.0

tco_results = pd.DataFrame({
    "S1": {
        bound: (
            capex["capex"]
            + opex_s1[bound]["opex"]
            - tax_impact
        )
        for bound in ["Lower", "Mean", "Upper"]
    },

    "S2": {
        bound: (
            capex["capex"]
            + opex_s2[bound]["opex"]
            - tax_impact
        )
        for bound in ["Lower", "Mean", "Upper"]
    },
}).T

tco_results

,Lower,Mean,Upper
S1,52670378.25,60325836.91,68043545.21
S2,40695392.09,45008279.45,49376658.43


Summary Reference Table

In [28]:
deterministic_summary = pd.DataFrame({
    "S1": {
        "Lifetime H2 [kg]": lifetime_h2,
        "Resource cost [EUR]": opex_s1["Mean"]["resource_cost"],
        "Labour cost [EUR]": opex_s1["Mean"]["labour_cost"],
        "EoL cost [EUR]": opex_s1["Mean"]["eol_cost"],
        "OPEX [EUR]": opex_s1["Mean"]["opex"],
        "TCO [EUR]": tco_results.loc["S1", "Mean"],
    },

    "S2": {
        "Lifetime H2 [kg]": lifetime_h2,
        "Resource cost [EUR]": opex_s2["Mean"]["resource_cost"],
        "Labour cost [EUR]": opex_s2["Mean"]["labour_cost"],
        "EoL cost [EUR]": opex_s2["Mean"]["eol_cost"],
        "OPEX [EUR]": opex_s2["Mean"]["opex"],
        "TCO [EUR]": tco_results.loc["S2", "Mean"],
    },
})

display(deterministic_summary)

,S1,S2
Lifetime H2 [kg],8898230.21,8898230.21
Resource cost [EUR],58670468.04,43352910.58
Labour cost [EUR],32907.15,32907.15
EoL cost [EUR],32388.38,32388.38
OPEX [EUR],58735763.57,43418206.11
TCO [EUR],60325836.91,45008279.45


## 9. Levelized cost of hydrogen

In [29]:
from util_model import (
    discounted_h2_production,
)

# Annual hydrogen production
annual_h2_production = (
    PEMWE_H2_PER_HALF_YEAR * 2.0
)

# Discount lifetime hydrogen production using the real WACC
# already calculated for the deterministic EoL model
discounted_h2 = discounted_h2_production(
    annual_h2_production=annual_h2_production,
    discount_rate=eol["real_wacc"],
    lifetime_years=LIFETIME_YEARS,
)

# Calculate deterministic LCOH for S1 and S2
lcoh_results = tco_results / discounted_h2

# Lifetime production consistency
assert np.isclose(
    annual_h2_production * LIFETIME_YEARS,
    lifetime_h2,
)

# Discounted production must be lower than nominal lifetime production
assert discounted_h2 < lifetime_h2

lcoh_results

,Lower,Mean,Upper
S1,8.43,9.66,10.89
S2,6.51,7.21,7.90


## 10. Lifecycle uncertainty analysis

### 10.1 Monte Carlo model setup
The Monte Carlo analysis uses the deterministic lower, mean, and upper resource-cost trajectories of each electricity-price scenario. A triangular distribution is defined for the total resource cost. Each sampled total is then mapped back to a 40-period resource-cost trajectory between the deterministic scenario paths.

In [30]:
from util_uncertainty import (
    resource_uncertainty_stats,
    resource_path_from_total,
)

# Build scenario-specific resource-cost uncertainty definitions.
s1_resource_uncertainty = resource_uncertainty_stats(
    resource_s1
)

s2_resource_uncertainty = resource_uncertainty_stats(
    resource_s2
)

resource_uncertainty_summary = pd.DataFrame({
    "S1": s1_resource_uncertainty,
    "S2": s2_resource_uncertainty,
})

resource_uncertainty_summary

,S1,S2
lower,74869738.20,55733504.01
mean,85753257.21,61554675.45
mode,85634303.21,61487543.41
upper,96755730.22,67442978.93
c,0.491847,0.491400


### 10.2 Full lifecycle Monte Carlo
Both scenarios use the same random seed. This keeps the material-price samples and random quantiles aligned between S1 and S2. The scenario difference therefore comes from the different resource-cost distributions.

> ### N_DRAWS set to 2048 instead of 350 

In [31]:
from util_uncertainty import (
    run_lcc_monte_carlo,
)


annual_h2_production = (
    PEMWE_H2_PER_HALF_YEAR * 2.0
)

mc_s1 = run_lcc_monte_carlo(
    scenario_name="S1",
    material_uncertainty=material_uncertainty,
    historical_stats=HISTORICAL_METAL_STATS,
    eol_assumptions=PEMWE_EOL_ASSUMPTIONS,
    resource_df=resource_s1,
    resource_stats=s1_resource_uncertainty,
    construction_labour_fraction=PEMWE_CONSTRUCTION_LABOUR_FRACTION,
    operation_labour_fraction=OPEX_LABOUR_FRACTION,
    deconstruction_fraction=PEMWE_DECONSTRUCTION_FRACTION,
    inflation_rate=INFLATION_RATE,
    equity=EQUITY,
    equity_return=EQUITY_RETURN,
    debt=DEBT,
    debt_interest_rate=DEBT_INTEREST_RATE,
    lifetime_years=LIFETIME_YEARS,
    annual_h2_production=annual_h2_production,
    eol_mode="PEMWE",
    n_draws=2048 * 5,
    seed=123456,
)

mc_s2 = run_lcc_monte_carlo(
    scenario_name="S2",
    material_uncertainty=material_uncertainty,
    historical_stats=HISTORICAL_METAL_STATS,
    eol_assumptions=PEMWE_EOL_ASSUMPTIONS,
    resource_df=resource_s2,
    resource_stats=s2_resource_uncertainty,
    construction_labour_fraction=PEMWE_CONSTRUCTION_LABOUR_FRACTION,
    operation_labour_fraction=OPEX_LABOUR_FRACTION,
    deconstruction_fraction=PEMWE_DECONSTRUCTION_FRACTION,
    inflation_rate=INFLATION_RATE,
    equity=EQUITY,
    equity_return=EQUITY_RETURN,
    debt=DEBT,
    debt_interest_rate=DEBT_INTEREST_RATE,
    lifetime_years=LIFETIME_YEARS,
    annual_h2_production=annual_h2_production,
    eol_mode="PEMWE",
    n_draws=2048 * 5,
    seed=123456,
)

### 10.3 Monte Carlo results

The lifecycle uncertainty results are summarized by the median and 5th–95th percentile interval of LCOH for each scenario.

In [32]:
mc_results = pd.concat(
    [mc_s1, mc_s2],
    ignore_index=True,
)

# Build the summary:

mc_lcoh_summary = (
    mc_results
    .groupby("scenario")["lcoh"]
    .agg(
        Mean="mean",
        Median="median",
        P05=lambda x: x.quantile(0.05),
        P95=lambda x: x.quantile(0.95),
    )
)

mc_lcoh_summary["Deterministic"] = pd.Series({
    "S1": lcoh_results.loc["S1", "Mean"],
    "S2": lcoh_results.loc["S2", "Mean"],
})

display(mc_lcoh_summary)

# And plot both distributions

plot_distribution(
    mc_s1["lcoh"],
    title="PEMWE LCOH uncertainty — S1",
    x_title="LCOH [EUR/kg H2]",
).show()

plot_distribution(
    mc_s2["lcoh"],
    title="PEMWE LCOH uncertainty — S2",
    x_title="LCOH [EUR/kg H2]",
).show()

,Mean,Median,P05,P95,Deterministic
scenario,,,,,
S1,9.69,9.69,8.85,10.55,9.66
S2,7.24,7.24,6.76,7.73,7.21


In [33]:
lcoh_plot = (
    mc_results
    .groupby("scenario")["lcoh"]
    .agg(
        Median="median",
        P05=lambda x: x.quantile(0.05),
        P95=lambda x: x.quantile(0.95),
    )
    .reset_index()
)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=lcoh_plot["scenario"],
    y=lcoh_plot["Median"],
    error_y={
        "type": "data",
        "symmetric": False,
        "array": (
            lcoh_plot["P95"]
            - lcoh_plot["Median"]
        ),
        "arrayminus": (
            lcoh_plot["Median"]
            - lcoh_plot["P05"]
        ),
    },
    name="Median LCOH",
))

fig.add_hline(
    y=4.0,
    line_dash="dash",
    annotation_text="4 EUR/kg H2 benchmark",
)

fig.update_layout(
    title="PEMWE levelized cost of hydrogen under S1 and S2",
    xaxis_title="Scenario",
    yaxis_title="LCOH [EUR/kg H2]",
)

fig.show()

In [ ]:
opex_plot = pd.DataFrame({
    "Scenario": ["S1", "S2"],
    "Lower": [
        opex_s1["Lower"]["opex"],
        opex_s2["Lower"]["opex"],
    ],
    "Mean": [
        opex_s1["Mean"]["opex"],
        opex_s2["Mean"]["opex"],
    ],
    "Upper": [
        opex_s1["Upper"]["opex"],
        opex_s2["Upper"]["opex"],
    ],
})

fig = go.Figure()

fig.add_trace(go.Bar(
    x=opex_plot["Scenario"],
    y=opex_plot["Mean"],
    error_y={
        "type": "data",
        "symmetric": False,
        "array": (
            opex_plot["Upper"]
            - opex_plot["Mean"]
        ),
        "arrayminus": (
            opex_plot["Mean"]
            - opex_plot["Lower"]
        ),
    },
    name="Lifecycle OPEX",
))

fig.update_layout(
    title="PEMWE lifecycle operating expenditure under S1 and S2",
    xaxis_title="Scenario",
    yaxis_title="Lifecycle OPEX [EUR]",
)

fig.show()

In [35]:
technology = "PEMWE" 

lcoh_summary = (
    mc_results
    .groupby("scenario")["lcoh"]
    .agg(
        Center="median",
        Lower=lambda x: x.quantile(0.05),
        Upper=lambda x: x.quantile(0.95),
    )
    .reset_index()
    .rename(columns={"scenario": "Scenario"})
)

lcoh_summary["Technology"] = technology
lcoh_summary["Metric"] = "LCOH"

opex_summary = pd.DataFrame({
    "Scenario": ["S1", "S2"],
    "Center": [
        opex_s1["Mean"]["opex"],
        opex_s2["Mean"]["opex"],
    ],
    "Lower": [
        opex_s1["Lower"]["opex"],
        opex_s2["Lower"]["opex"],
    ],
    "Upper": [
        opex_s1["Upper"]["opex"],
        opex_s2["Upper"]["opex"],
    ],
    "Technology": technology,
    "Metric": "OPEX",
})

pd.concat(
    [lcoh_summary, opex_summary],
    ignore_index=True,
).to_csv(f"{technology.lower()}_plot_summary.csv", index=False)

## 11. Threshold Analysis
### 11.1 Monte Carlo threshold-crossing probability

> *Implementation Note*: The old AWE Notebook generated Monnte Carlo LCOH distributions but did not explicitily calculate threshold distances or crossing frequencies. 

In [36]:
# Deployment-oriented LCOH thresholds.
thresholds = [
    3.5,
    4.0,
    4.5,
    5.0,
]

threshold_rows = []

for scenario, results in [
    ("S1", mc_s1),
    ("S2", mc_s2),
]:
    for threshold in thresholds:

        crossings = (
            results["lcoh"] <= threshold
        )

        n_crossings = int(
            crossings.sum()
        )

        crossing_probability = (
            crossings.mean()
        )

        threshold_rows.append({
            "Scenario": scenario,
            "Threshold [EUR/kg H2]": threshold,
            "Crossings": n_crossings,
            "Cases": len(results),
            "P_cross": crossing_probability,
            "P_cross [%]": crossing_probability * 100.0,
        })

threshold_results = pd.DataFrame(
    threshold_rows
)

threshold_results

,Scenario,Threshold [EUR/kg H2],Crossings,Cases,P_cross,P_cross [%]
0,S1,3.50,0,10240,0.000000,0.000000
1,S1,4,0,10240,0.000000,0.000000
2,S1,4.50,0,10240,0.000000,0.000000
3,S1,5,0,10240,0.000000,0.000000
4,S2,3.50,0,10240,0.000000,0.000000
5,S2,4,0,10240,0.000000,0.000000
6,S2,4.50,0,10240,0.000000,0.000000
7,S2,5,0,10240,0.000000,0.000000


### 11.2 Deterministic threshold distance

In [37]:
deterministic_threshold_rows = []

for scenario in ["S1", "S2"]:

    deterministic_lcoh = (
        lcoh_results.loc[
            scenario,
            "Mean",
        ]
    )

    for threshold in thresholds:

        deterministic_threshold_rows.append({
            "Scenario": scenario,
            "Threshold [EUR/kg H2]": threshold,
            "Deterministic LCOH [EUR/kg H2]":
                deterministic_lcoh,
            "Gap [EUR/kg H2]":
                deterministic_lcoh - threshold,
        })

deterministic_threshold_results = pd.DataFrame(
    deterministic_threshold_rows
)

deterministic_threshold_results

,Scenario,Threshold [EUR/kg H2],Deterministic LCOH [EUR/kg H2],Gap [EUR/kg H2]
0,S1,3.50,9.66,6.16
1,S1,4,9.66,5.66
2,S1,4.50,9.66,5.16
3,S1,5,9.66,4.66
4,S2,3.50,7.21,3.71
5,S2,4,7.21,3.21
6,S2,4.50,7.21,2.71
7,S2,5,7.21,2.21


In [38]:
# assert (
#     threshold_results["Cases"] == 2048
# ).all()

# assert (
#     threshold_results["P_cross"]
#     .between(0.0, 1.0)
#     .all()
# )

import plotly.express as px

# Visualize how the model-based crossing probability changes
# as the LCOH benchmark is relaxed.
fig = px.line(
    threshold_results,
    x="Threshold [EUR/kg H2]",
    y="P_cross [%]",
    color="Scenario",
    markers=True,
    title="PEMWE LCOH threshold-crossing probability",
)

fig.update_traces(
    line_width=2,
    marker_size=8,
)

fig.update_layout(
    xaxis_title="LCOH threshold [EUR/kg H2]",
    yaxis_title="Crossing probability [%]",
    hovermode="x unified",
)

fig.update_xaxes(
    tickmode="array",
    tickvals=thresholds,
)

fig.show()

## 12. Component-level stress tests
### 12.1 Stress-test calculation

The component stress test varies selected AWE component prices one at a time around the deterministic S2 baseline. All other component prices, OPEX, EoL terms, hydrogen production, and financial assumptions remain fixed.

Rectifier / Power Electronics, PTFE, and Zirfon Membrane are tested over price changes from −50% to +100%.

In [39]:
# Components explicitly evaluated in the PEMWE stress test.
stress_components = [
    "Iridium",
    "Rectifier / Power Electronics",
    "Nafion N117",
]

# One-at-a-time component-price shocks.
#
# -0.50 = 50% price reduction
#  0.00 = deterministic baseline
#  1.00 = doubling of the component price
stress_shocks = np.array([
    -0.50,
    -0.25,
     0.00,
     0.25,
     0.50,
     0.75,
     1.00,
])

# Central deployment-oriented comparison benchmark.
stress_threshold = 4.0

# Deterministic S2 baseline.
baseline_material_cost = float(material_cost)
baseline_capex = float(capex["capex"])
baseline_opex = float(opex_s2["Mean"]["opex"])
baseline_tco = float(tco_results.loc["S2", "Mean"])
baseline_lcoh = float(lcoh_results.loc["S2", "Mean"])

baseline_gap = (
    baseline_lcoh
    - stress_threshold
)

stress_rows = []

for component in stress_components:

    component_row = material_df.loc[
        material_df["Material"] == component
    ]

    if len(component_row) != 1:
        raise ValueError(
            f"Expected exactly one material row for {component}."
        )

    component_row = component_row.iloc[0]

    amount = float(
        component_row["Amount"]
    )

    baseline_unit_price = float(
        component_row["Unit Price"]
    )

    baseline_component_cost = (
        amount
        * baseline_unit_price
    )

    for shock in stress_shocks:

        # Change only the selected component price.
        stressed_unit_price = (
            baseline_unit_price
            * (1.0 + shock)
        )

        stressed_component_cost = (
            amount
            * stressed_unit_price
        )

        # Replace the baseline component cost with the stressed cost.
        stressed_material_cost = (
            baseline_material_cost
            - baseline_component_cost
            + stressed_component_cost
        )

        # Recalculate CAPEX because construction labour is defined
        # as a fraction of total material/component cost.
        stressed_capex_result = investment_cost(
            material_cost=stressed_material_cost,
            labour_fraction=PEMWE_CONSTRUCTION_LABOUR_FRACTION,
        )

        stressed_capex = float(
            stressed_capex_result["capex"]
        )

        # Stress-test scope:
        # keep deterministic S2 OPEX and all other lifecycle terms fixed.
        stressed_tco = (
            stressed_capex
            + baseline_opex
            - tax_impact
        )

        stressed_lcoh = (
            stressed_tco
            / discounted_h2
        )

        capex_change = (
            (stressed_capex / baseline_capex - 1.0)
            * 100.0
        )

        lcoh_change = (
            stressed_lcoh
            - baseline_lcoh
        )

        # define threshold-gap closure for
        # favorable component-price shocks.
        if shock < 0.0:
            gap_closure = (
                (baseline_lcoh - stressed_lcoh)
                / baseline_gap
                * 100.0
            )
        elif shock == 0.0:
            gap_closure = 0.0
        else:
            gap_closure = np.nan

        stress_rows.append({
            "Component": component,
            "Shock": shock,
            "Shock [%]": shock * 100.0,
            "Unit Price": stressed_unit_price,
            "Material Cost": stressed_material_cost,
            "CAPEX": stressed_capex,
            "CAPEX change [%]": capex_change,
            "TCO": stressed_tco,
            "LCOH [EUR/kg H2]": stressed_lcoh,
            "LCOH change [EUR/kg H2]": lcoh_change,
            "Threshold gap closure [%]": gap_closure,
        })

stress_results = pd.DataFrame(
    stress_rows
)

stress_results

,Component,Shock,Shock [%],Unit Price,Material Cost,CAPEX,CAPEX change [%],TCO,LCOH [EUR/kg H2],LCOH change [EUR/kg H2],Threshold gap closure [%]
0,Iridium,-0.500000,-50,28585.87,1034429.18,1249590.45,-21.41,44667796.56,7.15,-0.054505,1.70
1,Iridium,-0.250000,-25,42878.81,1175357.53,1419831.89,-10.71,44838038.00,7.18,-0.027253,0.850311
2,Iridium,0.000000,0.000000,57171.74,1316285.88,1590073.34,0.000000,45008279.45,7.21,0.000000,0.000000
3,Iridium,0.250000,25,71464.68,1457214.23,1760314.79,10.71,45178520.89,7.23,0.027253,NaN
4,Iridium,0.500000,50,85757.62,1598142.58,1930556.23,21.41,45348762.34,7.26,0.054505,NaN
5,Iridium,0.750000,75,100050.55,1739070.93,2100797.68,32.12,45519003.79,7.29,0.081758,NaN
6,Iridium,1,100,114343.49,1879999.28,2271039.13,42.83,45689245.23,7.31,0.109010,NaN
7,Rectifier / Power Electronics,-0.500000,-50,160349.50,1155936.38,1396371.15,-12.18,44814577.25,7.17,-0.031008,0.967491
8,Rectifier / Power Electronics,-0.250000,-25,240524.25,1236111.13,1493222.24,-6.09,44911428.35,7.19,-0.015504,0.483746
9,Rectifier / Power Electronics,0.000000,0.000000,320699,1316285.88,1590073.34,0.000000,45008279.45,7.21,0.000000,0.000000


### 12.2 Baseline consistency checks

In [40]:
# Every zero-shock case must reproduce the deterministic S2 baseline.
baseline_stress_rows = stress_results[
    np.isclose(
        stress_results["Shock"],
        0.0,
    )
]

assert np.allclose(
    baseline_stress_rows["Material Cost"],
    baseline_material_cost,
)

assert np.allclose(
    baseline_stress_rows["CAPEX"],
    baseline_capex,
)

assert np.allclose(
    baseline_stress_rows["TCO"],
    baseline_tco,
)

assert np.allclose(
    baseline_stress_rows["LCOH [EUR/kg H2]"],
    baseline_lcoh,
)

print(
    "Component stress-test baseline checks passed."
)

Component stress-test baseline checks passed.


### 12.3 Main stress-test results

In [41]:
stress_summary = (
    stress_results[
        stress_results["Shock [%]"].isin(
            [-50.0, 100.0]
        )
    ][
        [
            "Component",
            "Shock [%]",
            "CAPEX",
            "CAPEX change [%]",
            "LCOH [EUR/kg H2]",
            "LCOH change [EUR/kg H2]",
            "Threshold gap closure [%]",
        ]
    ]
    .reset_index(drop=True)
)

stress_summary

,Component,Shock [%],CAPEX,CAPEX change [%],LCOH [EUR/kg H2],LCOH change [EUR/kg H2],Threshold gap closure [%]
0,Iridium,-50,1249590.45,-21.41,7.15,-0.054505,1.70
1,Iridium,100,2271039.13,42.83,7.31,0.109010,NaN
2,Rectifier / Power Electronics,-50,1396371.15,-12.18,7.17,-0.031008,0.967491
3,Rectifier / Power Electronics,100,1977477.73,24.36,7.27,0.062016,NaN
4,Nafion N117,-50,1493812.65,-6.05,7.19,-0.015410,0.480797
5,Nafion N117,100,1782594.72,12.11,7.24,0.030819,NaN


### 12.4 LCOH response plot

In [42]:
shared_layout = {
    "font": {
        "size": 16,
    },
    "margin": {
        "t": 25,
        "r": 20,
        "b": 70,
        "l": 35,
    },
    "legend_title": "Legend",
    "legend": {
        "orientation": "v",
        "x": 0.05,
        "xanchor": "left",
        "y": -0.08,
        "yanchor": "top",
        "font": {
            "size": 14,
        },
    },
}
fig_stress_lcoh = px.line(
    stress_results,
    x="Shock [%]",
    y="LCOH [EUR/kg H2]",
    color="Component",
    markers=True,
    # title=(
    #     "PEMWE component-price stress test — "
    #     "LCOH response"
    # ),
)

# Central comparison benchmark.
fig_stress_lcoh.add_hline(
    y=stress_threshold,
    line_dash="dash",
    annotation_text="4.0 EUR/kg H2 benchmark",
)

# Deterministic S2 baseline.
fig_stress_lcoh.add_hline(
    y=baseline_lcoh,
    line_dash="dot",
    annotation_text="S2 baseline",
)
# Focus y-axis on the relevant LCOH range.
fig_stress_lcoh.update_yaxes(
    range=[7.3, 7.7],
)

# Save publication-ready SVG.
fig_stress_lcoh.write_image(
    "plots/PEMWE_component_stress_lcoh.svg",
    format="svg",
)


fig_stress_lcoh.update_layout(
    **shared_layout,
    xaxis_title="Component-price shock [%]",
    yaxis_title="LCOH [EUR/kg H2]",
    hovermode="x unified",
)

fig_stress_lcoh.update_xaxes(
    tickmode="array",
    tickvals=stress_shocks * 100.0,
)
# Focus y-axis on the relevant LCOH range.
fig_stress_lcoh.update_yaxes(
    range=[0, 7.7],
)

# Save publication-ready SVG.
fig_stress_lcoh.write_image(
    "plots/PEMWE_component_stress_lcoh.svg",
    format="svg",
)



fig_stress_lcoh.show()

### 12.5 CAPEX response plot

In [43]:
fig_stress_capex = px.line(
    stress_results,
    x="Shock [%]",
    y="CAPEX change [%]",
    color="Component",
    markers=True,

)

fig_stress_capex.add_hline(
    y=0.0,
    line_dash="dot",
)

fig_stress_capex.update_layout(
    **shared_layout,
    xaxis_title="Component-price change [%]",
    yaxis_title="CAPEX change [%]",
    hovermode="x unified",
)

fig_stress_capex.update_xaxes(
    tickmode="array",
    tickvals=stress_shocks * 100.0,
)

# Save publication-ready SVG.
fig_stress_capex.write_image(
    "plots/PEMWE_component_stress_capex.svg",
    format="svg",
)

fig_stress_capex.show()

### 12.6 Threshold-gap leverage

In [44]:
favorable_stress_results = (
    stress_results[
        np.isclose(
            stress_results["Shock"],
            -0.50,
        )
    ][
        [
            "Component",
            "CAPEX change [%]",
            "LCOH [EUR/kg H2]",
            "LCOH change [EUR/kg H2]",
            "Threshold gap closure [%]",
        ]
    ]
    .sort_values(
        "Threshold gap closure [%]",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(favorable_stress_results)

fig_gap_closure = px.bar(
    favorable_stress_results,
    x="Component",
    y="Threshold gap closure [%]",
    title=(
        "PEMWE threshold-gap closure under "
        "50% component-price reductions"
    ),
)

fig_gap_closure.update_layout(
    xaxis_title="Component",
    yaxis_title="4 EUR/kg H2 threshold gap closed [%]",
)

fig_gap_closure.show()

,Component,CAPEX change [%],LCOH [EUR/kg H2],LCOH change [EUR/kg H2],Threshold gap closure [%]
0,Iridium,-21.41,7.15,-0.054505,1.70
1,Rectifier / Power Electronics,-12.18,7.17,-0.031008,0.967491
2,Nafion N117,-6.05,7.19,-0.015410,0.480797
